# Task 1 — Gold-only đối chứng và Silver → Gold QLoRA

Notebook dùng **26 bài gold train / 6 bài gold validation** và 422 bài silver Task 1 sau kiểm tra chất lượng. Mỗi lần chạy tải mới hai ZIP từ [Drive](https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC), ghi SHA-256, tạo thư mục dữ liệu/output riêng theo run. Chỉ đánh giá và chọn adapter bằng **gold validation**. Chọn runtime **GPU** trong Colab.

Quy trình: cài thư viện → clone mới branch `task-1` → tải ZIP mới và chuẩn bị gold/silver → preflight token → smoke test (một bước silver, một bước gold và suy luận) → chạy đối chứng đầy đủ. Push code của branch `task-1` trước khi chạy notebook. ZIP cần quyền xem qua gdown; nếu Drive hạn chế tải, sửa quyền chia sẻ của hai file rồi chạy lại.

Các output đang hiển thị trong file notebook là kết quả chạy trước đó được giữ lại; hãy **Run all** để tạo báo cáo silver/gold mới.


In [3]:
%pip -q install "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown

In [4]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1'
from datetime import datetime, timezone
# Clone fresh on each run so an older Colab checkout with local edits is preserved.
REPO_DIR = Path('/content') / ('llm-final-project-task1-finetune-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
required_source = ['scripts/train_task1.py', 'src/task1/finetune.py', 'configs/task1/lora_7b.yaml', 'scripts/prepare_data.py', 'scripts/prepare_task1_silver.py', 'src/task1/silver_data.py']
missing_source = [name for name in required_source if not (REPO_DIR / name).is_file()]
if missing_source:
    raise FileNotFoundError(f'Branch task-1 trên GitHub chưa có code fine-tune: {missing_source}. Push branch rồi chạy lại.')
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path = [str(REPO_DIR)] + [p for p in sys.path if p != str(REPO_DIR)]
for module in list(sys.modules):
    if module == 'src' or module.startswith('src.'):
        del sys.modules[module]
print('Source:', REPO_DIR, '| branch:', REPO_BRANCH, '| commit:', commit)

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Source: /content/llm-final-project-task1-finetune-20261015T080906045051Z | branch: task-1 | commit: 4a65937


In [5]:
import gdown, hashlib, shutil, zipfile
from datetime import datetime, timezone
from src.data.loader import resolve_dataset_root

FOLDER_URL = 'https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC'
DRIVE_FILES = {
    'gold_public': {'id': '1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj', 'name': 'public-test-cham-diem-phan-hoi-files.zip'},
    'silver': {'id': '1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl', 'name': 'generate-dataset.zip'},
}
RUN_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DOWNLOAD_DIR = Path('/content') / f'task1-silver-gold-{RUN_TIME}'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=False)

# Always download both files into this run's empty directory; never reuse a same-name ZIP.
archives = {}
source_manifest = {'drive_folder': FOLDER_URL, 'downloaded_at_utc': RUN_TIME, 'git_commit': commit, 'files': {}}
for key, spec in DRIVE_FILES.items():
    path = DOWNLOAD_DIR / spec['name']
    downloaded = gdown.download(id=spec['id'], output=str(path), quiet=False)
    if not downloaded or not path.is_file() or not zipfile.is_zipfile(path):
        raise RuntimeError(f'Không tải được ZIP hợp lệ: {spec["name"]}')
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    with zipfile.ZipFile(path) as zf:
        bad = zf.testzip()
        if bad:
            raise ValueError(f'{spec["name"]} lỗi CRC ở {bad}')
    archives[key] = path
    source_manifest['files'][key] = {
        'drive_file_id': spec['id'], 'name': spec['name'],
        'sha256': digest, 'bytes': path.stat().st_size,
    }
    print(spec['name'], 'SHA-256:', digest)

def extract_safe(archive, destination):
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as zf:
        root = destination.resolve()
        for info in zf.infolist():
            target = (root / info.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'ZIP chứa đường dẫn không an toàn: {info.filename}')
        zf.extractall(destination)

def extract_nested(outer_path, inner_name, destination):
    with zipfile.ZipFile(outer_path) as outer:
        matches = [name for name in outer.namelist() if Path(name).name == inner_name]
        if len(matches) != 1:
            raise FileNotFoundError(f'ZIP ngoài phải có đúng một {inner_name}; thấy {matches}')
        inner = DOWNLOAD_DIR / inner_name
        with outer.open(matches[0]) as src, inner.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    if not zipfile.is_zipfile(inner):
        raise ValueError(f'ZIP lồng không hợp lệ: {inner_name}')
    source_manifest['files'][inner_name] = {
        'parent': 'gold_public', 'sha256': hashlib.sha256(inner.read_bytes()).hexdigest(),
        'bytes': inner.stat().st_size,
    }
    extract_safe(inner, destination)
    return resolve_dataset_root(destination)

GOLD_ROOT = extract_nested(archives['gold_public'], 'sample_dataset.zip', DOWNLOAD_DIR / 'gold')
PUBLIC_ROOT = extract_nested(archives['gold_public'], 'public_test_release.zip', DOWNLOAD_DIR / 'public')
extract_safe(archives['silver'], DOWNLOAD_DIR / 'silver')
SILVER_ROOT = resolve_dataset_root(DOWNLOAD_DIR / 'silver')
RUN_ID = RUN_TIME + '-' + source_manifest['files']['gold_public']['sha256'][:8] + '-' + source_manifest['files']['silver']['sha256'][:8]
DATA_HOME = PROJECT_HOME / 'data/task1_finetune_silver_gold' / RUN_ID
PROCESSED_DIR = DATA_HOME / 'gold_processed'
SPLITS_DIR = DATA_HOME / 'gold_splits'
SILVER_PROCESSED_DIR = DATA_HOME / 'silver_processed'
OUTPUT_DIR = PROJECT_HOME / 'data/outputs/task1/qwen25_coder7b_qlora_silver_gold' / RUN_ID
DATA_HOME.mkdir(parents=True, exist_ok=False)
SOURCE_MANIFEST = DATA_HOME / 'source_manifest.json'
SOURCE_MANIFEST.write_text(json.dumps(source_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
subprocess.run([sys.executable, str(REPO_DIR / 'scripts/prepare_data.py'),
                '--data_dir', str(GOLD_ROOT), '--output_dir', str(PROCESSED_DIR),
                '--splits_dir', str(SPLITS_DIR), '--seed', '42', '--val_ratio', '0.2'],
               cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, str(REPO_DIR / 'scripts/prepare_task1_silver.py'),
                '--silver-root', str(SILVER_ROOT), '--gold-root', str(GOLD_ROOT),
                '--public-root', str(PUBLIC_ROOT), '--output-dir', str(SILVER_PROCESSED_DIR)],
               cwd=REPO_DIR, check=True)
gold_manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text(encoding='utf-8'))
silver_manifest = json.loads((SILVER_PROCESSED_DIR / 'silver_manifest.json').read_text(encoding='utf-8'))
assert (gold_manifest['n_samples'], gold_manifest['n_train_ids'], gold_manifest['n_val_ids']) == (32, 26, 6), gold_manifest
assert (silver_manifest['n_source_task1'], silver_manifest['n_accepted'], silver_manifest['n_excluded']) == (462, 422, 40), silver_manifest
assert (silver_manifest['n_heldout_not_read'], silver_manifest['n_task3_only_not_read']) == (65, 150), silver_manifest
for part in ('train', 'val'):
    fixed_ids = json.loads((REPO_DIR / 'data/splits' / f'{part}_ids.json').read_text(encoding='utf-8'))
    current_ids = json.loads((SPLITS_DIR / f'{part}_ids.json').read_text(encoding='utf-8'))
    assert fixed_ids == current_ids, f'Gold {part} split IDs đã thay đổi'
print('Gold train/val:', 26, '/', 6, '| Silver accepted:', silver_manifest['n_accepted'])
print('Filtered silver audit:', SILVER_PROCESSED_DIR / 'silver_manifest.json')
print('Output:', OUTPUT_DIR)


Downloading...
From: https://drive.google.com/uc?id=1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj
To: /content/task1-silver-gold-20261015T080908454574Z/public-test-cham-diem-phan-hoi-files.zip
100%|██████████| 274k/274k [00:00<00:00, 99.0MB/s]


public-test-cham-diem-phan-hoi-files.zip SHA-256: 637a033ead19dc2fd5c5a82c859e92637762b09f4da870c2d012f108a9183dad


Downloading...
From: https://drive.google.com/uc?id=1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl
To: /content/task1-silver-gold-20261015T080908454574Z/generate-dataset.zip
100%|██████████| 566k/566k [00:00<00:00, 130MB/s]


generate-dataset.zip SHA-256: 88eff95f2841efd061ab0203907f9142ad6a2a2b72e266a4d55043a50aedac9c
Gold train/val: 26 / 6 | Silver accepted: 422
Filtered silver audit: /content/drive/MyDrive/LLM/Final Project/data/task1_finetune_silver_gold/20261015T080908454574Z-637a033e-88eff95f/silver_processed/silver_manifest.json
Output: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261015T080908454574Z-637a033e-88eff95f


In [6]:
RUNNER = REPO_DIR / 'scripts/train_task1.py'
CONFIG = REPO_DIR / 'configs/task1/lora_7b.yaml'

def run_task1(*flags):
    command = [sys.executable, str(RUNNER), '--config', str(CONFIG),
               '--processed-dir', str(PROCESSED_DIR), '--splits-dir', str(SPLITS_DIR),
               '--silver-processed-dir', str(SILVER_PROCESSED_DIR),
               '--source-manifest', str(SOURCE_MANIFEST),
               '--output-dir', str(OUTPUT_DIR), *flags]
    subprocess.run(command, cwd=REPO_DIR, check=True)

run_task1('--preflight')


## Smoke test và huấn luyện đầy đủ

Smoke test chạy **một bước gold-only, một bước silver, một bước gold sau silver**, rồi sinh dự đoán cho một bài gold validation. Lượt đầy đủ dùng 422 silver trong 1 epoch, sau đó 26 gold trong 3 epoch; baseline dùng 26 gold trong 3 epoch. Mỗi epoch gold được đánh giá trên đúng 6 gold validation. Chỉ lưu `best_adapter` của silver→gold nếu thắng gold-only theo QWK, hòa thì MAE; hòa hoàn toàn giữ gold-only. Mỗi lần chạy notebook có thư mục output riêng, không ghi đè kết quả cũ.


In [7]:
RUN_SMOKE = True
if RUN_SMOKE:
    run_task1('--smoke')
    smoke_summary = json.loads((OUTPUT_DIR / 'smoke/training_summary.json').read_text(encoding='utf-8'))
    print('Smoke test OK. Selected:', smoke_summary['selected_variant'])
    print('Gold validation QWK:', smoke_summary['selected_metrics']['qwk'])


Smoke test OK. Selected: gold_only
Gold validation QWK: 0.0


In [8]:
RUN_FULL = True
if RUN_FULL:
    run_task1()
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text(encoding='utf-8'))
    print('Selected variant:', summary['selected_variant'], '| epoch:', summary['selected_epoch'])
    print('Gold-only QWK/MAE:', summary['gold_only_best']['metrics']['qwk'], summary['gold_only_best']['metrics']['mae'])
    print('Silver→gold QWK/MAE:', summary['silver_then_gold_best']['metrics']['qwk'], summary['silver_then_gold_best']['metrics']['mae'])
    print('Selected valid output rate:', summary['selected_metrics']['valid_output_rate'])
    print('Adapter:', OUTPUT_DIR / 'best_adapter')
    print('Reports:', OUTPUT_DIR / 'gold_only/validation_report.json', OUTPUT_DIR / 'silver_then_gold/validation_report.json')
    print('Source hashes:', OUTPUT_DIR / 'source_manifest.json')


Selected variant: silver_then_gold | epoch: 2
Gold-only QWK/MAE: 0.4298401420959148 3.1666666666666665
Silver→gold QWK/MAE: 0.5871559633027523 2.3333333333333335
Selected valid output rate: 1.0
Adapter: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261015T080908454574Z-637a033e-88eff95f/best_adapter
Reports: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261015T080908454574Z-637a033e-88eff95f/gold_only/validation_report.json /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261015T080908454574Z-637a033e-88eff95f/silver_then_gold/validation_report.json
Source hashes: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_silver_gold/20261015T080908454574Z-637a033e-88eff95f/source_manifest.json
